# Competitor Agent peer 비교 워크스루

이 노트북은 팀원이 Competitor Agent의 peer 비교 흐름과 출력 구조를 실험적으로 확인하기 위한 문서입니다. production 코드는 `src/stock_agent/tools/peer_tool.py`와 `src/stock_agent/agents/competitor.py`에 있으며, 이 노트북은 해당 코드를 대체하지 않습니다.

주요 확인 목표는 다음과 같습니다.

- 샘플 `CompanyPeer`, `PriceSnapshot`, `FinancialSnapshot` 입력이 `PeerMetricRow`로 변환되는 방식 확인
- `calculate_relative_position`이 target과 peer 지표를 비교해 점수와 상대 위치를 만드는 방식 확인
- 실제 DB 연결 흐름에서는 `run_competitor`가 `build_peer_comparison`을 호출하고, DB가 없거나 실패하면 fallback 결과가 나올 수 있음을 확인

## 샘플 입력 설명

아래 예시는 삼성전자를 target으로 두고 같은 섹터의 peer 3개를 비교하는 최소 샘플입니다. 각 회사에 대해 가격 스냅샷과 최근/전년 재무 스냅샷을 만들어 `calculate_metric_row`에 전달합니다.

- `CompanyPeer`: 회사 식별자, 종목코드, 회사명, 섹터
- `PriceSnapshot`: 기준일, 종가, 시가총액, 거래량
- `FinancialSnapshot`: 사업연도, 매출, 영업이익, 순이익, 자본, 부채

숫자는 흐름 확인용 예시이며 실제 투자 판단용 데이터가 아닙니다.

In [ ]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from stock_agent.schemas.analysis import AgentState, CuratorResult, Portfolio, UserProfile
from stock_agent.tools.peer_tool import (
    CompanyPeer,
    FinancialSnapshot,
    PriceSnapshot,
    calculate_metric_row,
    calculate_relative_position,
)
from stock_agent.agents.competitor import run_competitor


In [ ]:
def make_metric_row(
    company: CompanyPeer,
    market_cap: int,
    close_price: int,
    revenue: int,
    operating_income: int,
    net_income: int,
    equity: int,
    liabilities: int,
    previous_revenue: int,
):
    price = PriceSnapshot(
        stock_code=company.stock_code,
        base_date="2026-05-25",
        close_price=close_price,
        market_cap=market_cap,
        volume=1_000_000,
    )
    latest = FinancialSnapshot(
        corp_code=company.corp_code,
        bsns_year=2025,
        revenue=revenue,
        operating_income=operating_income,
        net_income=net_income,
        equity=equity,
        liabilities=liabilities,
    )
    previous = FinancialSnapshot(
        corp_code=company.corp_code,
        bsns_year=2024,
        revenue=previous_revenue,
        operating_income=None,
        net_income=None,
        equity=None,
        liabilities=None,
    )
    return calculate_metric_row(company, price, latest, previous)


target_company = CompanyPeer(
    corp_code="00126380",
    stock_code="005930",
    corp_name="삼성전자",
    sector="semiconductor",
)

peer_companies = [
    CompanyPeer(corp_code="00164779", stock_code="000660", corp_name="SK하이닉스", sector="semiconductor"),
    CompanyPeer(corp_code="00126447", stock_code="000990", corp_name="DB하이텍", sector="semiconductor"),
    CompanyPeer(corp_code="00104856", stock_code="042700", corp_name="한미반도체", sector="semiconductor"),
]

target_row = make_metric_row(
    target_company,
    market_cap=420_000_000_000_000,
    close_price=70_000,
    revenue=300_000_000_000_000,
    operating_income=30_000_000_000_000,
    net_income=21_000_000_000_000,
    equity=280_000_000_000_000,
    liabilities=70_000_000_000_000,
    previous_revenue=250_000_000_000_000,
)

peer_rows = [
    make_metric_row(
        peer_companies[0],
        market_cap=150_000_000_000_000,
        close_price=180_000,
        revenue=70_000_000_000_000,
        operating_income=18_000_000_000_000,
        net_income=12_000_000_000_000,
        equity=90_000_000_000_000,
        liabilities=45_000_000_000_000,
        previous_revenue=55_000_000_000_000,
    ),
    make_metric_row(
        peer_companies[1],
        market_cap=3_000_000_000_000,
        close_price=52_000,
        revenue=1_200_000_000_000,
        operating_income=220_000_000_000,
        net_income=180_000_000_000,
        equity=2_000_000_000_000,
        liabilities=900_000_000_000,
        previous_revenue=1_000_000_000_000,
    ),
    make_metric_row(
        peer_companies[2],
        market_cap=8_000_000_000_000,
        close_price=130_000,
        revenue=650_000_000_000,
        operating_income=190_000_000_000,
        net_income=160_000_000_000,
        equity=900_000_000_000,
        liabilities=250_000_000_000,
        previous_revenue=520_000_000_000,
    ),
]

metric_rows = [target_row, *peer_rows]
print(json.dumps([row.model_dump() for row in metric_rows], ensure_ascii=False, indent=2))


In [ ]:
position = calculate_relative_position(metric_rows, target_stock_code=target_row.stock_code)
print(json.dumps(position.model_dump(), ensure_ascii=False, indent=2))


## 실제 DB 연결 흐름

실제 에이전트 실행에서는 `run_competitor`가 `AgentState.curator`의 종목코드와 섹터를 읽고, DB 연결을 얻은 뒤 `build_peer_comparison`을 호출합니다. `build_peer_comparison`은 target 회사, peer 후보, 가격, 재무 데이터를 조회해 metric row를 만들고 최종 `CompetitorResult`에 들어갈 비교 결과를 구성합니다.

로컬에서 DB가 준비되지 않았거나 `psycopg` 같은 연결 의존성이 없으면 `run_competitor`는 예외를 잡아 fallback-safe mock 결과를 반환할 수 있습니다. 따라서 아래 셀의 출력에 `mock_data_fallback` 또는 `fallback` 문구가 보이면 실제 peer DB 비교가 아니라 안전용 예시 결과로 해석해야 합니다.

In [ ]:
state = AgentState(
    user_query="삼성전자 경쟁사와 비교해줘",
    user_profile=UserProfile(user_id="notebook-demo", risk_tolerance="medium"),
    portfolio=Portfolio(),
    curator=CuratorResult(
        intent="stock_analysis",
        corp_name="삼성전자",
        stock_code="005930",
        corp_code="00126380",
        sector="semiconductor",
    ),
)

# DB가 없거나 연결에 실패하면 run_competitor가 mock fallback 결과를 채울 수 있습니다.
result_state = run_competitor(state)
competitor_result = result_state.competitor

if competitor_result is None:
    print("Competitor 결과가 생성되지 않았습니다.")
else:
    result = competitor_result.model_dump()
    print("DB 연결 실패 시 mock fallback 결과가 포함될 수 있습니다.")
    for key in [
        "warnings",
        "data_quality_flags",
        "peer_summary",
        "score",
        "peer_selection_summary",
        "relative_position",
        "peers",
    ]:
        print(f"\n## {key}")
        print(json.dumps(result.get(key), ensure_ascii=False, indent=2))


## 결과 해석 체크리스트

결과를 볼 때는 아래 순서로 확인합니다.

1. `warnings`: peer 수 부족, 섹터 누락, fallback 여부처럼 결과 해석 전에 알아야 할 경고가 있는지 확인합니다.
2. `data_quality_flags`: target 또는 peer 지표 계산에 빠진 값이 있는지 확인합니다. 예를 들어 PER, PBR, ROE, 매출 성장률, 영업이익률, 부채비율 관련 flag가 있으면 해당 지표 해석을 보수적으로 봅니다.
3. `peer_summary`: 사람이 읽는 요약 문장으로 전체 비교 상황과 데이터 품질 메시지를 먼저 파악합니다.
4. `score`: 0~100 범위의 종합 점수입니다. 점수만 단독으로 보지 말고 앞의 경고와 데이터 품질 flag를 함께 확인합니다.

추가로 `relative_position`은 valuation, ROE, 성장, 마진, 재무 안정성 percentile을 담고, `a1_peer_multiple_payload`는 A1 peer multiple 흐름에서 재사용할 median PER/PBR 정보를 담습니다.